<b>Data Mining Assignment: </b> <br> <br>
Student Name: Sushant Tuladhar <br>
Student ID: 23514047 <br>
Course Name: Data Mining <br>
Course ID: COMP50009 <br>

Import the following headers as shown below : <br>
<ol>
  <li> Numpy </li>
  <li> Pandas </li>
  <li> Sqlite3 </li>
</ol>

In [ ]:
import numpy as np
import pandas as pd
import sqlite3

Download the file from the github repository which I hosted to not load the file each time during run and downloads it automatically.

In [ ]:
#Download the file from the github repository in the colab file
!wget https://github.com/sushant-tuladhar/MachineLearningClassification/raw/refs/heads/main/Assignment2025S2.sqlite -O Assignment2025S2.sqlite

In [ ]:
connection = sqlite3.connect('Assignment2025S2.sqlite')

df_train = pd.read_sql_query("SELECT * FROM train", connection)
df_train.isna().sum() [df_train.isna().sum() > 0]

In [ ]:
df_train['Greenness_Index']=df_train['Greenness_Index'].fillna(df_train['Greenness_Index'].median())
df_train['Days_Since_Burn']=df_train['Days_Since_Burn'].fillna(df_train['Days_Since_Burn'].median())
df_train['Latitude']=df_train['Latitude'].fillna(df_train['Latitude'].median())

In [ ]:
df_train.isna().sum()[df_train.isna().sum() > 0]

In [ ]:
df_train.drop('index',inplace=True,axis=1)

In [ ]:
df_train.duplicated().sum()

In [ ]:
df_train=df_train.drop_duplicates(keep='first').reset_index(drop=True)

In [ ]:
df_train.shape

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

numeric_df=df_train.select_dtypes(include='number')

numeric_df.hist(bins=20,figsize=(12,12),edgecolor='k')
plt.suptitle('Distribution of Numerical figures', fontsize=20)
plt.show()

In [ ]:
#Categorical data frame

categorical_df= df_train.select_dtypes(include=['object','category'])
# print(categorical_df)
for col in categorical_df.columns:
    plt.figure(figsize=(10,10))
    sns.countplot(x=col, data=categorical_df, palette='pastel', order=categorical_df[col].value_counts().index)
    plt.title(f"Distribution of {col} Categories", fontsize=20)
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
df_test=pd.read_sql_query("Select * FROM test", connection)
df_test.head()

In [ ]:
df_test.shape

In [ ]:
df_test.isna().sum()[df_test.isna().sum() > 0]

In [ ]:
df_test.drop('index',inplace=True,axis=1)
df_test.head()
df_test.duplicated().sum()

In [ ]:
df_test.shape

In [ ]:
 #Do the fillment and do the testing for the test class as well if there are any empty or duplicated cases

target='class'
columns = ['Human_Disturbance','Soil_Type','Burn_Season','Surface_Water_Presence','Landform']
cross_tab=pd.crosstab(df_train[col],df_train[target])
for col in columns:
    plt.figure(figsize=(10,10))
    sns.heatmap(cross_tab, annot = True, cmap='YlGnBu', fmt='g')
    plt.title(f"{col} vs {target}")
    plt.show()

In [ ]:
X_train=df_train.drop('class',axis=1)
Y_train=df_train['class']

#Separate numeric and categorical data for preprocessing
numeric_features= X_train.select_dtypes(include='number').columns.tolist()
categorical_features=X_train.select_dtypes(include=['object','category']).columns.tolist()

In [ ]:
from sklearn.preprocessing import LabelEncoder

for col in categorical_features:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])

In [ ]:
X_train.shape

In [ ]:
Y_train.shape

In [ ]:
le1= LabelEncoder()
Y_train=le1.fit_transform(Y_train)

In [ ]:
le1.classes_

In [ ]:
np.unique(Y_train)

In [ ]:
#Using StratifiedKFold Model
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

rf_model=RandomForestClassifier(n_estimators=100, random_state=42)
skf=StratifiedKFold(n_splits=10, random_state=42,shuffle=True)

cv_scores=cross_val_score(rf_model,X_train,Y_train,cv=skf, scoring='accuracy')

print("Cross-validation accuracy for each fold:", cv_scores)
print("Mean CV Accuracy:", np.mean(cv_scores))
print("CV Standard Deviation:", np.std(cv_scores))

In [ ]:
rf_model.fit(X_train,Y_train)

In [ ]:
#Data_test duplication and index drop already done
X_test=df_test.drop('class',axis=1)
Y_test=df_test['class']

numeric_features= X_test.select_dtypes(include='number').columns.tolist()
categorical_features=X_test.select_dtypes(include=['object','category']).columns.tolist()

In [ ]:
X_test.shape

In [ ]:
for col in categorical_features:
    le= LabelEncoder()
    X_test[col] = le.fit_transform(X_test[col])

In [ ]:
X_test.head()

In [ ]:
#Assuming the rfmodel is already trained
Y_pred= rf_model.predict(X_test)

#If you also want prediction probabilities for classification
Y_prob= rf_model.predict_proba(X_test)[:,1]

In [ ]:
Y_pred

In [ ]:
y_pred_labels = le1.inverse_transform(Y_pred)

In [ ]:
y_pred_labels

Using OneHotEncoder and using decision tree to check the accuracy of the model

In [ ]:
#Here prepare the model as done in the practicals
X_train1=df_train.drop('class',axis=1)
Y_train1=df_train['class']

X_test1=df_test.drop('class',axis=1)
Y_test1=df_test['class']

numeric_features=X_train1.select_dtypes(include='number').columns.tolist()
from sklearn.preprocessing import StandardScaler
#
scaler=StandardScaler()
scaler.fit(X_train1[numeric_features])
#
X_train1[numeric_features]=scaler.fit(X_train1[numeric_features])
X_test1[numeric_features]=scaler.fit(X_test1[numeric_features])

In [ ]:
X_train1.describe()

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Choose the categorical values
categorical_cols = X_train1.select_dtypes(include='object').columns.tolist()

# # Build an encoder from the training data
encoder = OneHotEncoder(drop='first',             # Remove one of the categories from each feature (it's redundant)
                        handle_unknown='ignore',  # If there are categories in the test data that are not in the train data, don't complain, just set all values to 0
                        )
encoder.fit(X_train1[categorical_cols])

# # Apply the encoder to the train/test data, turning it into a pandas data frame
X_train_1hot = pd.DataFrame(encoder.transform(X_train1[categorical_cols]).toarray(), columns=encoder.get_feature_names_out())
X_train_1hot.set_index(X_train1.index, inplace=True)

# Drop the categorical columns and replace them with the one hot encoded ones
X_train1.drop(columns=categorical_cols, inplace=True)
X_train1 = pd.concat([X_train1, X_train_1hot], axis=1)

# Do the above again for the test data (apply encoder, replace columns)
X_test_1hot = pd.DataFrame(encoder.transform(X_test1[categorical_cols]).toarray(), columns=encoder.get_feature_names_out())
X_test_1hot.set_index(X_test1.index, inplace=True)
X_test1.drop(columns=categorical_cols, inplace=True)
X_test1 = pd.concat([X_test1, X_test_1hot], axis=1)

In [ ]:
X_train1

In [ ]:
#Another model building for testing the accuracy
from sklearn.tree import DecisionTreeClassifier, plot_tree
#Create a Decision tree with maximum of 2 levels
dt_model= DecisionTreeClassifier(max_depth=2)
dt_model.fit(X_train1,Y_train1)
y_hat=dt_model.predict(X_test1)

y_hat_prob=dt_model.predict_proba(X_test1)[:,1]


In [ ]:
fig,ax = plt.subplots(1,1,figsize=(10,10))
plot_tree(dt_model,filled=True, ax=ax, fontsize=10)
plt.show()

In [ ]:
from sklearn.model_selection import StratifiedKFold, KFold, ShuffleSplit

In [ ]:
# This is random sampling
ss = ShuffleSplit(n_splits=10, test_size=15, random_state=4)
# This is non-random sampling, we just break the data in to 10 contiguous sub-sets
kf = KFold(n_splits=10)
# Ensuring the balance between classes in the model/validate sets
# means we should use stratified sampling
skf = StratifiedKFold(n_splits=10)


In [ ]:
# This cell sets up a nice visulisation that I found on the scikit-learn documentation page.
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.preprocessing import LabelEncoder
cmap_data = plt.cm.Paired
cmap_cv = plt.cm.coolwarm

def plot_cv_indices(cv, X, y, ax, n_splits, lw=10):
    """
    Create a sample plot for indices of a cross-validation object.
    Adapted from https://scikit-learn.org/stable/auto_examples/model_selection/plot_cv_indices.html#define-a-function-to-visualize-cross-validation-behavior

    Parameters
    ----------
    cv: cross validation method

    X : training data

    y : data labels

    ax : matplolib axes object

    n_splits : number of splits

    lw : line width for plotting
    """

    # Turn labels into integers
    le = LabelEncoder()
    y = le.fit_transform(y)


    # Generate the training/testing visualizations for each CV split
    for ii, (tr, tt) in enumerate(cv.split(X=X, y=y)):
        # Fill in indices with the training/test groups
        indices = np.array([np.nan] * len(X))
        indices[tt] = 1
        indices[tr] = 0

        # Visualize the results
        ax.scatter(range(len(indices)), [ii + .5] * len(indices),
                   c=indices, marker='_', lw=lw, cmap=cmap_cv,
                   vmin=-.2, vmax=1.2)

    # Plot the data classes at the end
    ax.scatter(range(len(X)), [ii + 1.5] * len(X),
               c=y, marker='_', lw=lw, cmap=cmap_data)

    # Formatting
    yticklabels = list(range(n_splits)) + ['class']
    ax.set(yticks=np.arange(n_splits+1) + .5, yticklabels=yticklabels,
           xlabel='Sample index', ylabel="CV iteration",
           ylim=[n_splits+1.2, -.2])
    ax.set_title('{}'.format(type(cv).__name__), fontsize=15)
    return ax

In [ ]:
# Set up a figure with three subplots
fig, ax = plt.subplots(1,3, figsize=(18,6))
# visualise the KFolds algorithm
plot_cv_indices(kf,
                X_train1, Y_train1,
                ax=ax[0],
                n_splits=10)
# visualise the ShulffleSplit algorithm
plot_cv_indices(ss,
                X_train1, Y_train1,
                ax=ax[1],
                n_splits=10)
# visualise the StratifiedKFolds algorithm
plot_cv_indices(skf,
                X_train1, Y_train1,
                ax=ax[2],
                n_splits=10)
plt.show()

In [ ]:
cv_scores=cross_val_score(dt_model,X_train1,Y_train1,cv=skf, scoring='accuracy')
print("Cross Validation for Decision Tree Classifier \n")
print("Cross-validation accuracy for each fold:", cv_scores)
print("Mean CV Accuracy:", np.mean(cv_scores))
print("CV Standard Deviation:", np.std(cv_scores))

In [ ]:
# from sklearn.metrics import classification_report
# print(classification_report(Y_test1, y_hat))

Use K-NN classifier for finding the prediction <br>


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X_train2 = df_train.drop('class', axis=1)
Y_train2 = df_train['class']

X_test2 = df_test.drop('class', axis=1)
Y_test2 = df_test['class']

numeric_features = X_train2.select_dtypes(include='number').columns.tolist()
categorical_features = X_train2.select_dtypes(include=['object', 'category']).columns.tolist()

# Create a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
    ],
    remainder='passthrough' # Keep other columns
)

# Create a KNN model pipeline
knn_model = Pipeline(steps=[('preprocessor', preprocessor),
                            ('classifier', KNeighborsClassifier(n_neighbors=3))])

# Fit the KNN model
knn_model.fit(X_train2, Y_train2)

# Make predictions
y_pred = knn_model.predict(X_test2)

# If you also want prediction probabilities for classification
# Y_prob = knn_model.predict_proba(X_test2)[:,1] # Note: KNN predict_proba might not be reliable for all cases

In [ ]:
y_pred

In [ ]:
cv_scores=cross_val_score(knn_model,X_train2,Y_train2,cv=skf, scoring='accuracy')
print("Cross Validation Score for KNN method \n")
print("Cross-validation accuracy for each fold:", cv_scores)
print("Mean CV Accuracy:", np.mean(cv_scores))
print("CV Standard Deviation:", np.std(cv_scores))

Naive Bayes Prediction Algorithm:
<br>

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X_train3 = df_train.drop('class', axis=1)
Y_train3 = df_train['class']

X_test3 = df_test.drop('class', axis=1)
Y_test3 = df_test['class']

numeric_features = X_train3.select_dtypes(include='number').columns.tolist()
categorical_features = X_train3.select_dtypes(include=['object', 'category']).columns.tolist()

# Create a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
    ],
    remainder='passthrough' # Keep other columns
)

# Create a Naive Bayes model pipeline
nb_model = Pipeline(steps=[('preprocessor', preprocessor),
                            ('classifier', GaussianNB())])

# Fit the Naive Bayes model
nb_model.fit(X_train3, Y_train3)

# Make predictions
y_pred2 = nb_model.predict(X_test3)

# If you also want prediction probabilities for classification
# Y_prob = nb_model.predict_proba(X_test3)[:, 1]

In [ ]:
y_pred2

In [ ]:
cv_scores=cross_val_score(nb_model,X_train3,Y_train3,cv=skf, scoring='accuracy')
print("Cross Validation score for Naive Bayes Algorithm \n")
print("Cross-validation accuracy for each fold:", cv_scores)
print("Mean CV Accuracy:", np.mean(cv_scores))
print("CV Standard Deviation:", np.std(cv_scores))

So when we look into the error codes we Cross validation score to determine the accuracy of the code we see that the mean of accuracy are shown as below:

<ol>
  <li>Random Forest Classifier : 0.8995959595959595 </li>
  <li>Decision Tree Classifier : 0.6408080808080807 </li>
  <li>K-NN classifier: 0.8507070707070709 </li>
  <li> Naive Bayes Classifier: 0.35898989898989897</li>
</ol>

<br> Note: All the models were trained using StratifiedKFold for these different classification methods and the accuracy is as shown below for mean accuracy.

In [ ]:
#Do this at the last after the comparison has been done. So according to the the mean of accuracy we take and write the datas for Random Forest Classifier and K-NN classifier
pred_df=pd.DataFrame({
    "Predict1": y_pred_labels, #Random Forest Classifier
    "Predict2": y_pred, #K-NN Classifier
})

#Set index from 5000
pred_df.index= range(5000, 5000+len(pred_df))

#save to CSV file
pred_df.to_csv("Answers_23514047.csv",index_label="index")

In [ ]:
#Check if the file is present or not for the prediction.csv and display them

with open('Answers_23514047.csv','r') as file:
    print(file.read())